<font color=red>This is not a complete solution but gives you some ideas how one could appraoch the problems.</font>

<font color=red>
Common problems:
    
- Integral vs. vector norm
- random sign of numerical eigenstates -> consider plotting probability density
- Missing discussion and interpretation of results.
</font>

# Programing exercise 1: Single particle in a 1D potential

Due on Monday, 21.10.2024, 20h

## Defining the problem

We want to calculate the eigenenergies and eigenfunction of a quantum particle in a on-dimensional potential, i.e. solve the eigenvalue problem

$$\left[-\frac{1}{2} \partial_{x}^2 + V(x)\right] \phi(x) = E \phi(x)$$

by representing the wave function $\phi(x)$ on a discrete spatial grid.

In [ ]:
# load standard libraries

import numpy as np   # standard numerics library
import matplotlib.pyplot as plt   # for making plots
import Comp_Quant_Dynam as cqd

#%matplotlib inline

### Exercise 1

Warmup: Consider the harmonic oscillator $V(x)=\frac{1}{2}x^2$. Plot the analytical solutions of the eigenstates on a spatial grid. Plot the lowest 6 eigenfunctions. Useful functions: scipy.special.hermite(), np.linspace(). Always use numpy arrays!

In [ ]:
# define the grid
L = 10
npoints = 401
xvals = np.linspace(-L/2,L/2,npoints)

In [ ]:
# plot single wave function (or probability density)
n_plot=0;
H0_eigenstate = cqd.hamiltonians.HO_eigenstates_exact(n_plot, xvals)

# plot exact
plt.figure()
plt.plot(xvals, H0_eigenstate)
plt.title("exact analytic")
plt.xlabel("x")
plt.ylabel("$\\phi_n(x)$")
plt.show()

In [ ]:
# plot probability density for 6 lowest eigenstates

# plot exact
plt.figure()
plt.plot(xvals,xvals**2/2,'k')
for n_plot_itr in range(0,6):
    plt.plot(xvals,1 / 2 + n_plot_itr + np.abs(cqd.hamiltonians.HO_eigenstates_exact(n_plot_itr,xvals) ** 2))
plt.title("exact analytic")
plt.xlabel("x")
plt.ylabel("$|\\phi_n(x)|^2$")
plt.ylim([0,6])
plt.show()

### Exercise 2

Solve the eigenproblem using numpy.linalg.eigh().
Use a grid size of 20 length units and 401 gridpoints as a test case and plot the lowest 6 eigenfunctions.

Hints: eigh() is for Hermitian matrices, which have real eigenvalues. It returns the eigenvalues and eigenvectors already sorted with increasing eigenvalues. You could also use numpy.linalg.eig(), which also works for matrices with in general complex eigenvalues, but then you would have to sort them to get the ground state and lowest lying excited states, e.g. using np.argsort(). Remember that the eigenvectors are the *columns* of the array returned by eigh()!

Useful functions: np.diag(), np.ones()

Optional: Use an interactive function to look at the eigenfunctions systematically. The following example helps you with this. It allows you to click through the eigenfunctions one by one.

In [ ]:
from ipywidgets import interactive, fixed

g = lambda x, k: np.sin(k * x) * x # test function for plotting

interactive_plot = interactive(cqd.plotting.plot_func, func = fixed(g), k=(0.0,3))
interactive_plot

In [ ]:
from numpy import linalg as LA   # matrix diagonalization and more linear algebra functions

Set up the problem and solve it using eigh().

In [ ]:
# define the grid
L = 20
npoints = 400
dx = L/(npoints - 1)
xvals = np.linspace(-L/2,L/2,npoints)

# build the terms of the Hamiltonian
H_pot = cqd.hamiltonians.HO_potential(xvals)
H_kin = cqd.hamiltonians.H_kinetic(xvals)

H_mat = H_pot + H_kin

# diagonalize: here the whole magic is happening
evals, evecs = LA.eigh(H_mat)


# # same thing with eig()
# evals, evecs = LA.eig(H_mat)

# # sort the eigenvalues and eigenfunctions in ascending order:
# indOrder = np.argsort(evals)
# evalsSrt = evals[indOrder]
# evecsSrt = evecs[:,indOrder]

Print the resulting lowest eigenvalues and plot eigenfunctions

In [ ]:
print(evals[0:10])

n_plot = 1
plt.plot(xvals,np.abs(evecs[:,n_plot])**2/dx)
plt.xlabel("x")
plt.ylabel("$|\\phi_n(x)|^2$")
plt.show()

In [ ]:
def plotWF(n):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(xvals,np.abs(evecs[:,n])**2/dx)
    ax.set_xlim(-5, 5)
    ax.set_ylim(0, 0.6)
    # add labels and legends
    ax.set_title("$E_n=$"+str(evals[n]))
    ax.set_xlabel("$x$")
    ax.set_ylabel("$|\\phi_n(x)|^2$")
    plt.show()

In [ ]:
interactive_plot = interactive(cqd.plotting.plot_eigenstate, n=(0, 10), x = fixed(xvals), evals = fixed(evals), evecs = fixed(evecs))
interactive_plot

### Exercise 3 (optional)

Having shown that the numerical diagonalization qualitatively gives the desired result, we now want to analyse its precision more quantitatively.

Check the convergence of the numerical solutions with respect to grid size and spacing. Compare the eigenvalues and eigenfunction to the analytical solutions.

Which eigenvalue has the smallest numerical error and why?

Calculate the ground state energy for different grid sizes and grid spacings and compare to the analytical solution. A systematic analysis is not necessary. Just check what happens if you use a too small grid or too few grid points.

Compare the numerically obtained eigenfunctions to the analytical solution by plotting them on top of each other. You will notice that they are normalized differently. Why is this? How to normlize the eigenvectors obtained by exact diagonalization "properly"? Plot the difference between the probability densities of the analytical and (properly normalized) numerical solution.

In [ ]:
# define the grid
L = 10
npoints = 401
dx = L/(npoints - 1)
xvals = np.linspace(-L/2,L/2,npoints)

# build the terms of the Hamiltonian
H_pot = cqd.hamiltonians.HO_potential(xvals)
H_kin = cqd.hamiltonians.H_kinetic(xvals)

H_mat = H_pot + H_kin

# diagonalize: here the whole magic is happening
evals, evecs = LA.eigh(H_mat)

In [ ]:
#check that the state is normalized (if we want the integral to be normalized we have to divide by dx)
n_plot = 0
LA.norm(evecs[:,n_plot])

In [ ]:
# check convergence: relative error of eigenvalues
evals_exact = cqd.hamiltonians.HO_eigenenergies_exact(np.arange(evals.size))
(evals - evals_exact)[0:20]

In [ ]:
n_plot = 1

# compare exact to numerical
plt.figure()
plt.plot(xvals, np.abs(cqd.hamiltonians.HO_eigenstates_exact(n_plot, xvals)) ** 2, label = "exact")
plt.plot(xvals, np.abs(evecs[:,n_plot])**2 / dx, label = "numerical")
plt.title("exact and numerical")
plt.xlabel("x")
plt.ylabel("$|\\phi_n(x)|^2$")
plt.legend()

# plot relative error
plt.figure()
plt.plot(xvals, np.abs(np.abs(cqd.hamiltonians.HO_eigenstates_exact(n_plot, xvals)) ** 2 - np.abs(evecs[:, n_plot]) ** 2 / dx))
plt.title("absolute probability error")
plt.xlabel("x")
plt.ylabel("error");

Observations:

If the grid is too small, the ground state wave function gets narrower due to the hard wall boundary conditions. All the eigenenergies lie above the exact ones.

If too few gridpoints are used, the curvature of the wave function at the center is not well resolved. The numerically obtained energies are below the exact ones, probably because the kinetic term is underestimated.

### Exercise 4 (optional)

We can now use our routine to calculate bound states in any kind of potentials, where no analytical solutions are known. As an example, consider the double well potential
$$
V(x)=-\frac{1}{2}x^2 + \lambda x^4
$$
where $\lambda$ controls the height of the barrier between the wells. 

Calculate eigenvalues and eigenfunction for some values of lambda. Plot the lowest few eigenstates and interpret your results. 

In [ ]:
L = 10
npoints = 401
dx = L/(npoints - 1)
xvals = np.linspace(-L / 2, L / 2, npoints)
 
# potential
Vvec = -0.5 * xvals ** 2 + 0.05 * xvals ** 4 # double well

# plot potential
plt.plot(xvals,Vvec)
plt.ylim(-2,5)
plt.xlabel("x")
plt.ylabel("V(x)")

H_pot_dw = np.diag(Vvec)
H_kin = cqd.hamiltonians.H_kinetic(xvals)

H_mat_dw = H_pot_dw + H_kin

# diagonalize: here the whole magic is happening
evals_dw, evecs_dw = LA.eigh(H_mat_dw)

In [ ]:
print(evals_dw[0:20])
#plt.plot(xvals,evecs[:,0]/dx)
#plt.plot(xvals,evecs[:,3]/dx)
plt.plot(xvals, np.abs(evecs_dw[:,0] ** 2) / dx, label = "ground state")
plt.plot(xvals, np.abs(evecs_dw[:,1] ** 2) / dx, label = "1st excited state")
plt.xlabel("x")
plt.ylabel("$|\\phi_n(x)|^2$")
plt.legend();

For small $\lambda$ (high barrier) the low lying eigenvalues of the double well come in pairs. The corresponding eigenfunctions are pairs with different symmetry. Their envelopes look quite similar. This means that there are two almost independent HOs that are weakly tunnel coupled.